In [1]:
%%configure -f
{
  "conf": {
    "spark.master": "yarn",
    "spark.speculation": "false",

    "spark.driver.cores": "8",
    "spark.driver.memory": "384g",
    "spark.driver.memoryOverhead": "96g",

    "spark.yarn.am.cores": "6",
    "spark.yarn.am.memory": "384g",
    "spark.yarn.am.memoryOverhead": "96g",

    "spark.driver.maxResultSize": "20g",

    "spark.executor.cores": "4",
    "spark.executor.memory": "18g",
    "spark.executor.memoryOverhead": "8g",

    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": "189",
    "spark.dynamicAllocation.initialExecutors": "189",
    "spark.dynamicAllocation.maxExecutors": "1000",
    "spark.dynamicAllocation.executorIdleTimeout": "60s",
    "spark.dynamicAllocation.cachedExecutorIdleTimeout": "300s",
    "spark.dynamicAllocation.schedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.sustainedSchedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.executorAllocationRatio": "1.0",

    "spark.locality.wait": "1s",

    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "false",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "268435456",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.sql.shuffle.partitions": "1024",
    "spark.default.parallelism": "1024",

    "spark.network.timeout": "800s",
    "spark.executor.heartbeatInterval": "60s",
    "spark.kryoserializer.buffer.max": "1g",
    "spark.rpc.message.maxSize": "1024",

    "spark.hadoop.fs.s3a.aws.credentials.provider": "com.amazonaws.auth.DefaultAWSCredentialsProviderChain",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryo.registrator": "is.hail.kryo.HailKryoRegistrator",

    "spark.hadoop.fs.s3.maxConnections": "50000",
    "spark.hadoop.fs.s3.connection.timeout": "120000",
    "spark.hadoop.fs.s3.socket.timeout": "120000",
    "spark.hadoop.fs.s3.maxRetries": "20",
    "spark.hadoop.fs.s3a.threads.max": "256",
    "spark.hadoop.fs.s3a.connection.maximum": "50000",
    "spark.hadoop.fs.s3a.connection.timeout": "120000",
    "spark.hadoop.fs.s3a.socket.timeout": "120000",
    "spark.hadoop.fs.s3a.attempts.maximum": "20",
    "spark.hadoop.fs.s3a.retry.interval": "1000",
    "spark.hadoop.fs.s3a.fast.upload": "true",
    "spark.hadoop.fs.s3a.multipart.size": "104857600",
    "spark.hadoop.fs.s3a.threads.keepalivetime": "60000",
    "spark.hadoop.fs.s3a.connection.establish.timeout": "30000",
    "spark.hadoop.fs.s3a.multipart.purge.age": "86400000"
  },
  "executorCores": 4,
  "executorMemory": "18G",
  "driverCores": 8,
  "driverMemory": "384G"
}

In [2]:
# Initialize Hail against the running SparkContext from Livy/EMR Notebooks
import hail as hl
hl.init(sc, log="/tmp/hail.log")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1782100276285_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/usr/local/lib/python3.11/site-packages/hail/backend/spark_backend.py:76: UserWarning: Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jar
  warnings.warn(
Running on Apache Spark version 3.5.5-amzn-1
SparkUI available at http://ip-192-168-102-190.ap-southeast-1.compute.internal:39583
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail.log

In [6]:
# source
vds_prefix = "s3://precise-scratch/goypav/SG10K_Health/VDS/"
annotated_vds_prefix = vds_prefix + "step1/annotated_vds/"
vds_step2_base_prefix = vds_prefix + "step2/batch4_7321/"

# input
manifest_uri = "s3://precise-scratch/goypav/SG10K_Health/VDS/vds_step1_manifest_SG10K_Health_batch4_7321.txt"

# output



FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
# -----------------------------
# 1. Read manifest of VDS paths
# -----------------------------
# # ### create vds_step1_manifest.txt
# aws s3 ls s3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/ \
# | awk '$NF ~ /^batch_20260617_16[0-9]{4}_[0-9]+\.vds\/?$/ {
#     print "s3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/" $NF
# }' \
# | sort \
# > vds_step1_manifest_SG10K_Health_batch4_7321.txt

# # ### copy to s3
# aws s3 cp vds_step1_manifest_SG10K_Health_batch4_7321.txt s3://precise-scratch/goypav/SG10K_Health/VDS/ --dryrun

# ============================================================
# SETUP CELL: read manifest and define 8 chunks
# ============================================================

import time
import traceback
import gc
from datetime import datetime

fs = hl.current_backend().fs

with fs.open(manifest_uri) as f:
    vds_paths = [line.strip() for line in f if line.strip()]

print(f"Found {len(vds_paths):,} VDSes")
print(vds_paths[:5])

# Split into 8 notebook-cell chunks:
# 7 chunks of 1000, last chunk of 321
chunk_size = 1000

chunks = [
    vds_paths[i:i + chunk_size]
    for i in range(0, len(vds_paths), chunk_size)
]

print(f"Number of chunks/cells: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"cell_idx={i}: {len(chunk):,} VDSes")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Found 7,321 VDSes
['s3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260617_163347_0000.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260617_163347_0001.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260617_163347_0002.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260617_163348_0003.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260617_163348_0004.vds/']
Number of chunks/cells: 8
cell_idx=0: 1,000 VDSes
cell_idx=1: 1,000 VDSes
cell_idx=2: 1,000 VDSes
cell_idx=3: 1,000 VDSes
cell_idx=4: 1,000 VDSes
cell_idx=5: 1,000 VDSes
cell_idx=6: 1,000 VDSes
cell_idx=7: 321 VDSes

In [8]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 0

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running cell_idx=0
Input VDSes in this cell: 1,000
Number of 100-sample batches: 10
Output prefix: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_00/
Logs prefix: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_00/logs/
  batch_0000: 100 VDSes
  batch_0001: 100 VDSes
  batch_0002: 100 VDSes
  batch_0003: 100 VDSes
  batch_0004: 100 VDSes
  batch_0005: 100 VDSes
  batch_0006: 100 VDSes
  batch_0007: 100 VDSes
  batch_0008: 100 VDSes
  batch_0009: 100 VDSes

In [9]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[2026-06-22 12:35:34] Starting cell_00/batch_0000
[2026-06-22 12:35:34] Number of input VDSes: 100
[2026-06-22 12:35:34] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_00/batch_0000.vds
9700
[2026-06-22 13:23:24] Completed cell_00/batch_0000
[2026-06-22 13:23:24] Runtime: 2870.4 seconds (47.84 minutes)
332
82914
[2026-06-22 13:23:24] Starting cell_00/batch_0001
[2026-06-22 13:23:24] Number of input VDSes: 100
[2026-06-22 13:23:24] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_00/batch_0001.vds
9700
[2026-06-22 14:10:52] Completed cell_00/batch_0001
[2026-06-22 14:10:52] Runtime: 2847.7 seconds (47.46 minutes)
332
87449
[2026-06-22 14:10:52] Starting cell_00/batch_0002
[2026-06-22 14:10:52] Number of input VDSes: 100
[2026-06-22 14:10:52] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_00/batch_0002.vds
9700
[2026-06-22 15:01:51] Completed cell_00/batch_0002
[2026-06-22 15:01:51] Runtime: 

In [10]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 1

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running cell_idx=1
Input VDSes in this cell: 1,000
Number of 100-sample batches: 10
Output prefix: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_01/
Logs prefix: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch4_7321/cell_01/logs/
  batch_0000: 100 VDSes
  batch_0001: 100 VDSes
  batch_0002: 100 VDSes
  batch_0003: 100 VDSes
  batch_0004: 100 VDSes
  batch_0005: 100 VDSes
  batch_0006: 100 VDSes
  batch_0007: 100 VDSes
  batch_0008: 100 VDSes
  batch_0009: 100 VDSes

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 2

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 3

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 4

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 5

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 6

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

In [ ]:
# ============================================================
# WORKER CELL
# Change only this number: 0, 1, 2, 3, 4, 5, 6, or 7
# ============================================================

cell_idx = 7

# Each worker cell handles ~1000 VDSes,
# internally combined in batches of 100.
cell_vds_paths = chunks[cell_idx]

batch_size = 100

vds_step2_prefix = f"{vds_step2_base_prefix}cell_{cell_idx:02d}/"
logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    cell_vds_paths[i:i + batch_size]
    for i in range(0, len(cell_vds_paths), batch_size)
]

print(f"Running cell_idx={cell_idx}")
print(f"Input VDSes in this cell: {len(cell_vds_paths):,}")
print(f"Number of 100-sample batches: {len(vds_batches)}")
print(f"Output prefix: {vds_step2_prefix}")
print(f"Logs prefix: {logs_prefix}")

for i, batch in enumerate(vds_batches):
    print(f"  batch_{i:04d}: {len(batch):,} VDSes")

In [ ]:
# ============================================================
# WORKER LOOP
# Run after setting cell_idx above
# ============================================================

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting cell_{cell_idx:02d}/{batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(
            input_vdses
        )

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED cell_{cell_idx:02d}/{batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed / 60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

## check output

In [ ]:
for batch_idx, batch_paths in enumerate(vds_batches):

    vds_uri = f"{vds_step2_prefix}batch_{batch_idx:04d}.vds"

    try:
        vds = hl.vds.read_vds(vds_uri)

        expected = len(batch_paths)
        observed = vds.n_samples()

        status = "✓" if expected == observed else "✗"

        print(
            f"{status} batch_{batch_idx:04d}: "
            f"expected={expected}, observed={observed}"
        )

    except Exception as e:
        print(f"✗ batch_{batch_idx:04d}: FAILED ({e})")

In [ ]:
# # -----------------------------
# # Read first combined batch VDS
# # -----------------------------

# vds_batch1_uri = f"{vds_step2_prefix}batch_0000.vds"

# vds = hl.vds.read_vds(vds_batch1_uri)

# print(f"Loaded: {vds_batch1_uri}")

In [ ]:
# # -----------------------------
# # Inspect schemas
# # -----------------------------

# print("=== Reference data ===")
# vds.reference_data.describe()


In [ ]:
# print("\n=== Variant data ===")
# vds.variant_data.describe()

In [ ]:
# # -----------------------------
# # Basic summary
# # -----------------------------

# print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
# print(f"Number of samples: {vds.n_samples():,}")
# print(f"Total number of variants: {vds.variant_data.count_rows():,}")
# print(f"Variant partitions: {vds.variant_data.n_partitions():,}")
# print(f"Reference partitions: {vds.reference_data.n_partitions():,}")